# Create Retail Ontology

Create a Microsoft Fabric ontology item from core retail Silver/Gold tables and realtime Eventhouse tables.

## Included business entity types
- Geography
- Store
- DistributionCenter
- Truck
- Customer
- Product
- Receipt
- OnlineOrder
- Promotion
- Payment
- CustomerSegment (gold / ML)
- ChurnPrediction (gold / ML)

Realtime Eventhouse tables are bound as additional TimeSeries bindings on the
same business entities where their keys align. For example, Receipt binds to the
Silver `fact_receipts` table and to Eventhouse `receipt_created` /
`receipt_line_added`; Payment binds to Silver `fact_payments` and Eventhouse
`payment_processed`.

## Included relationships
- Store -> Geography
- DistributionCenter -> Geography
- Customer -> Geography
- Truck -> DistributionCenter
- Receipt -> Customer
- Receipt -> Store
- Receipt -> Product (via receipt lines)
- OnlineOrder -> Customer
- OnlineOrder -> Product (via order lines)
- Promotion -> Receipt
- Promotion -> Store
- Promotion -> Customer
- Payment -> Receipt
- Payment -> OnlineOrder
- CustomerSegment -> Customer
- ChurnPrediction -> Customer
- Store -> Product stockout risk (via gold stockout_risk)

Where possible, relationships have both Lakehouse contextualizations (Silver/Gold
historical tables) and Eventhouse contextualizations (realtime KQL tables), so the
ontology stays a business relationship map rather than a graph of event-log rows.

By default the notebook deletes any ontology with the same name before recreating it.


In [ ]:
import base64
import json
import os
import random
import re
import time
import uuid

import sempy.fabric as fabric

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


def get_env_bool(var_name, default='true'):
    value = get_env(var_name, default)
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y'}


def normalize_ontology_name(value, default='RetailOntology_AutoGen'):
    raw_value = str(value or default).strip()
    normalized = re.sub(r'[^A-Za-z0-9_]+', '_', raw_value)
    normalized = re.sub(r'_+', '_', normalized).strip('_')
    if not normalized:
        normalized = default
    if not normalized[0].isalpha():
        normalized = f'RetailOntology_{normalized}'
    normalized = normalized[:89].rstrip('_')
    if not normalized:
        normalized = default
    if raw_value != normalized:
        print(
            f"Normalized ONTOLOGY_NAME from {raw_value!r} to {normalized!r} "
            'to satisfy Fabric item naming rules.'
        )
    return normalized


LAKEHOUSE_NAME = get_env('LAKEHOUSE_NAME', 'retail_lakehouse')
EVENTHOUSE_NAME = get_env(
    'EVENTHOUSE_NAME',
    get_env('KQL_DATABASE_NAME', 'retail_eventhouse'),
)
KQL_DATABASE_NAME = get_env('KQL_DATABASE_NAME', EVENTHOUSE_NAME)
KUSTO_URI = get_env('KUSTO_URI', get_env('EVENTHOUSE_QUERY_SERVICE_URI', ''))
SILVER_DB = get_env('SILVER_DB', 'silver')
GOLD_DB = get_env('GOLD_DB', 'gold')
BRONZE_DB = get_env('BRONZE_DB', 'cusn')
ONTOLOGY_NAME = normalize_ontology_name(get_env('ONTOLOGY_NAME', 'RetailOntology_AutoGen'))
ONTOLOGY_DESCRIPTION = get_env(
    'ONTOLOGY_DESCRIPTION',
    'Retail domain ontology generated from Lakehouse Silver and Eventhouse realtime tables.'
)
DELETE_EXISTING = get_env_bool('DELETE_EXISTING', 'true')
DELETE_WAIT_SECONDS = int(get_env('DELETE_WAIT_SECONDS', '30'))

print(f'SILVER_DB={SILVER_DB}')
print(f'GOLD_DB={GOLD_DB}')
print(f'BRONZE_DB={BRONZE_DB}')
print(f'LAKEHOUSE_NAME={LAKEHOUSE_NAME or "<auto-detect>"}')
print(f'EVENTHOUSE_NAME={EVENTHOUSE_NAME or "<auto-detect>"}')
print(f'KQL_DATABASE_NAME={KQL_DATABASE_NAME or "<auto-detect>"}')
print(f'KUSTO_URI={KUSTO_URI or "<auto-resolve>"}')
print(f'ONTOLOGY_NAME={ONTOLOGY_NAME}')
print(f'DELETE_EXISTING={DELETE_EXISTING}')
print(f'DELETE_WAIT_SECONDS={DELETE_WAIT_SECONDS}')


In [ ]:
# =============================================================================
# RETAIL ENTITY / RELATIONSHIP MODEL
# =============================================================================

# Ontology entities are business concepts. Realtime Eventhouse tables are
# additional TimeSeries bindings on those same entities when an event table
# carries the entity key.

ENTITY_CONFIG = {'dim_geographies': {'entity_name': 'Geography',
                     'key_candidates': ['geography_id', 'GeographyID', 'id', 'ID'],
                     'display_candidates': ['city',
                                            'City',
                                            'region',
                                            'Region',
                                            'state',
                                            'State',
                                            'country',
                                            'Country',
                                            'id',
                                            'ID']},
 'dim_stores': {'entity_name': 'Store',
                'key_candidates': ['store_id', 'StoreID', 'id', 'ID'],
                'display_candidates': ['store_number',
                                       'StoreNumber',
                                       'store_name',
                                       'StoreName',
                                       'address',
                                       'Address',
                                       'id',
                                       'ID']},
 'dim_distribution_centers': {'entity_name': 'DistributionCenter',
                              'key_candidates': ['dc_id', 'DCID', 'id', 'ID'],
                              'display_candidates': ['dc_number',
                                                     'DCNumber',
                                                     'name',
                                                     'Name',
                                                     'address',
                                                     'Address',
                                                     'id',
                                                     'ID']},
 'dim_trucks': {'entity_name': 'Truck',
                'key_candidates': ['truck_id', 'TruckID', 'id', 'ID'],
                'display_candidates': ['license_plate',
                                       'LicensePlate',
                                       'truck_number',
                                       'TruckNumber',
                                       'id',
                                       'ID']},
 'dim_customers': {'entity_name': 'Customer',
                   'key_candidates': ['customer_id', 'CustomerID', 'id', 'ID'],
                   'display_candidates': ['customer_name',
                                          'CustomerName',
                                          'loyalty_card',
                                          'LoyaltyCard',
                                          'email',
                                          'Email',
                                          'phone',
                                          'Phone',
                                          'id',
                                          'ID']},
 'dim_products': {'entity_name': 'Product',
                  'key_candidates': ['product_id', 'ProductID', 'id', 'ID'],
                  'display_candidates': ['product_name',
                                         'ProductName',
                                         'name',
                                         'Name',
                                         'brand',
                                         'Brand',
                                         'id',
                                         'ID']},
 'fact_receipts': {'entity_name': 'Receipt',
                   'key_candidates': ['receipt_id_ext',
                                      'ReceiptIdExt',
                                      'receipt_id',
                                      'ReceiptId',
                                      'ReceiptID'],
                   'display_candidates': ['receipt_id_ext',
                                          'ReceiptIdExt',
                                          'receipt_id',
                                          'ReceiptId',
                                          'ReceiptID']},
 'fact_online_order_headers': {'entity_name': 'OnlineOrder',
                               'key_candidates': ['order_id_ext',
                                                  'OrderIdExt',
                                                  'order_id',
                                                  'OrderId'],
                               'display_candidates': ['order_id_ext',
                                                      'OrderIdExt',
                                                      'order_id',
                                                      'OrderId']},
 'fact_promotions': {'entity_name': 'Promotion',
                     'key_candidates': ['receipt_id_ext', 'ReceiptId'],
                     'display_candidates': ['promo_code', 'PromoCode']},
 'fact_payments': {'entity_name': 'Payment',
                   'key_candidates': ['transaction_id', 'payment_id', 'TransactionId'],
                   'display_candidates': ['payment_method',
                                          'PaymentMethod',
                                          'transaction_id']},
 'customer_segments': {'entity_name': 'CustomerSegment',
                       'schema_db': 'gold',
                       'key_candidates': ['customer_id'],
                       'display_candidates': ['segment_label',
                                              'cluster_id',
                                              'customer_id']},
 'churn_predictions': {'entity_name': 'ChurnPrediction',
                       'schema_db': 'gold',
                       'key_candidates': ['customer_id'],
                       'display_candidates': ['risk_category', 'customer_id']}}

RELATIONSHIPS = [{'name': 'StoreLocatedInGeography',
  'source_table': 'dim_stores',
  'source_entity': 'Store',
  'source_column_candidates': ['store_id', 'StoreID', 'id', 'ID'],
  'target_entity': 'Geography',
  'target_column_candidates': ['geography_id', 'GeographyID']},
 {'name': 'DistributionCenterLocatedInGeography',
  'source_table': 'dim_distribution_centers',
  'source_entity': 'DistributionCenter',
  'source_column_candidates': ['dc_id', 'DCID', 'id', 'ID'],
  'target_entity': 'Geography',
  'target_column_candidates': ['geography_id', 'GeographyID']},
 {'name': 'CustomerLocatedInGeography',
  'source_table': 'dim_customers',
  'source_entity': 'Customer',
  'source_column_candidates': ['customer_id', 'CustomerID', 'id', 'ID'],
  'target_entity': 'Geography',
  'target_column_candidates': ['geography_id', 'GeographyID']},
 {'name': 'TruckAssignedToDistributionCenter',
  'source_table': 'dim_trucks',
  'source_entity': 'Truck',
  'source_column_candidates': ['truck_id', 'TruckID', 'id', 'ID'],
  'target_entity': 'DistributionCenter',
  'target_column_candidates': ['dc_id', 'DCID']},
 {'name': 'ReceiptPlacedByCustomer',
  'source_table': 'fact_receipts',
  'source_entity': 'Receipt',
  'source_column_candidates': ['receipt_id_ext',
                               'ReceiptIdExt',
                               'receipt_id',
                               'ReceiptId',
                               'ReceiptID'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id', 'CustomerID']},
 {'name': 'ReceiptAtStore',
  'source_table': 'fact_receipts',
  'source_entity': 'Receipt',
  'source_column_candidates': ['receipt_id_ext',
                               'ReceiptIdExt',
                               'receipt_id',
                               'ReceiptId',
                               'ReceiptID'],
  'target_entity': 'Store',
  'target_column_candidates': ['store_id', 'StoreID']},
 {'name': 'ReceiptContainsProduct',
  'source_table': 'fact_receipt_lines',
  'source_entity': 'Receipt',
  'source_column_candidates': ['receipt_id_ext',
                               'ReceiptIdExt',
                               'receipt_id',
                               'ReceiptId',
                               'ReceiptID'],
  'target_entity': 'Product',
  'target_column_candidates': ['product_id', 'ProductID']},
 {'name': 'OnlineOrderPlacedByCustomer',
  'source_table': 'fact_online_order_headers',
  'source_entity': 'OnlineOrder',
  'source_column_candidates': ['order_id_ext', 'OrderIdExt', 'order_id', 'OrderId'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id', 'CustomerID']},
 {'name': 'OnlineOrderContainsProduct',
  'source_table': 'fact_online_order_lines',
  'source_entity': 'OnlineOrder',
  'source_column_candidates': ['order_id_ext', 'OrderIdExt', 'order_id', 'OrderId'],
  'target_entity': 'Product',
  'target_column_candidates': ['product_id', 'ProductID']},
 {'name': 'PromotionAppliedToReceipt',
  'source_table': 'fact_promotions',
  'source_entity': 'Promotion',
  'source_column_candidates': ['receipt_id_ext', 'ReceiptId'],
  'target_entity': 'Receipt',
  'target_column_candidates': ['receipt_id_ext', 'ReceiptId']},
 {'name': 'PromotionAtStore',
  'source_table': 'fact_promotions',
  'source_entity': 'Promotion',
  'source_column_candidates': ['receipt_id_ext', 'ReceiptId'],
  'target_entity': 'Store',
  'target_column_candidates': ['store_id', 'StoreID']},
 {'name': 'PromotionByCustomer',
  'source_table': 'fact_promotions',
  'source_entity': 'Promotion',
  'source_column_candidates': ['receipt_id_ext', 'ReceiptId'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id', 'CustomerID']},
 {'name': 'PaymentForReceipt',
  'source_table': 'fact_payments',
  'source_entity': 'Payment',
  'source_column_candidates': ['transaction_id', 'payment_id', 'TransactionId'],
  'target_entity': 'Receipt',
  'target_column_candidates': ['receipt_id_ext', 'ReceiptId']},
 {'name': 'PaymentForOnlineOrder',
  'source_table': 'fact_payments',
  'source_entity': 'Payment',
  'source_column_candidates': ['transaction_id', 'payment_id', 'TransactionId'],
  'target_entity': 'OnlineOrder',
  'target_column_candidates': ['order_id_ext', 'order_id', 'OrderId']},
 {'name': 'CustomerHasSegment',
  'source_table': 'customer_segments',
  'schema_db': 'gold',
  'source_entity': 'CustomerSegment',
  'source_column_candidates': ['customer_id'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id']},
 {'name': 'CustomerHasChurnPrediction',
  'source_table': 'churn_predictions',
  'schema_db': 'gold',
  'source_entity': 'ChurnPrediction',
  'source_column_candidates': ['customer_id'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id']},
 {'name': 'StoreHasStockoutRiskForProduct',
  'source_table': 'stockout_risk',
  'schema_db': 'gold',
  'source_entity': 'Store',
  'source_column_candidates': ['store_id'],
  'target_entity': 'Product',
  'target_column_candidates': ['product_id']}]

EVENTHOUSE_ENTITY_BINDINGS = [{'entity_name': 'Receipt',
  'source_table': 'receipt_created',
  'source_column_candidates': ['receipt_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Receipt',
  'source_table': 'receipt_line_added',
  'source_column_candidates': ['receipt_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Payment',
  'source_table': 'payment_processed',
  'source_column_candidates': ['transaction_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Promotion',
  'source_table': 'promotion_applied',
  'source_column_candidates': ['receipt_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'OnlineOrder',
  'source_table': 'online_order_created',
  'source_column_candidates': ['order_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'OnlineOrder',
  'source_table': 'online_order_picked',
  'source_column_candidates': ['order_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'OnlineOrder',
  'source_table': 'online_order_shipped',
  'source_column_candidates': ['order_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'receipt_created',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'payment_processed',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'promotion_applied',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'inventory_updated',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'stockout_detected',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'reorder_triggered',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'customer_entered',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'customer_zone_changed',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'ble_ping_detected',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'truck_arrived',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'truck_departed',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'store_opened',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Store',
  'source_table': 'store_closed',
  'source_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Customer',
  'source_table': 'receipt_created',
  'source_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Customer',
  'source_table': 'payment_processed',
  'source_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Customer',
  'source_table': 'promotion_applied',
  'source_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Customer',
  'source_table': 'online_order_created',
  'source_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Product',
  'source_table': 'receipt_line_added',
  'source_column_candidates': ['product_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Product',
  'source_table': 'inventory_updated',
  'source_column_candidates': ['product_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Product',
  'source_table': 'stockout_detected',
  'source_column_candidates': ['product_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'Product',
  'source_table': 'reorder_triggered',
  'source_column_candidates': ['product_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'DistributionCenter',
  'source_table': 'inventory_updated',
  'source_column_candidates': ['dc_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'DistributionCenter',
  'source_table': 'stockout_detected',
  'source_column_candidates': ['dc_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'DistributionCenter',
  'source_table': 'reorder_triggered',
  'source_column_candidates': ['dc_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'DistributionCenter',
  'source_table': 'truck_arrived',
  'source_column_candidates': ['dc_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']},
 {'entity_name': 'DistributionCenter',
  'source_table': 'truck_departed',
  'source_column_candidates': ['dc_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True,
  'timestamp_candidates': ['ingest_timestamp']}]

EVENTHOUSE_RELATIONSHIP_CONTEXTS = [{'name': 'ReceiptPlacedByCustomer',
  'source_table': 'receipt_created',
  'source_entity': 'Receipt',
  'source_column_candidates': ['receipt_id'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'ReceiptAtStore',
  'source_table': 'receipt_created',
  'source_entity': 'Receipt',
  'source_column_candidates': ['receipt_id'],
  'target_entity': 'Store',
  'target_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'ReceiptContainsProduct',
  'source_table': 'receipt_line_added',
  'source_entity': 'Receipt',
  'source_column_candidates': ['receipt_id'],
  'target_entity': 'Product',
  'target_column_candidates': ['product_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'PaymentForReceipt',
  'source_table': 'payment_processed',
  'source_entity': 'Payment',
  'source_column_candidates': ['transaction_id'],
  'target_entity': 'Receipt',
  'target_column_candidates': ['receipt_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'PaymentForOnlineOrder',
  'source_table': 'payment_processed',
  'source_entity': 'Payment',
  'source_column_candidates': ['transaction_id'],
  'target_entity': 'OnlineOrder',
  'target_column_candidates': ['order_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'PromotionAppliedToReceipt',
  'source_table': 'promotion_applied',
  'source_entity': 'Promotion',
  'source_column_candidates': ['receipt_id'],
  'target_entity': 'Receipt',
  'target_column_candidates': ['receipt_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'PromotionAtStore',
  'source_table': 'promotion_applied',
  'source_entity': 'Promotion',
  'source_column_candidates': ['receipt_id'],
  'target_entity': 'Store',
  'target_column_candidates': ['store_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'PromotionByCustomer',
  'source_table': 'promotion_applied',
  'source_entity': 'Promotion',
  'source_column_candidates': ['receipt_id'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True},
 {'name': 'OnlineOrderPlacedByCustomer',
  'source_table': 'online_order_created',
  'source_entity': 'OnlineOrder',
  'source_column_candidates': ['order_id'],
  'target_entity': 'Customer',
  'target_column_candidates': ['customer_id'],
  'source': 'eventhouse',
  'schema_db': 'bronze',
  'optional': True}]

EXCLUDED_SOURCE_COLUMNS = {'__index_level_0__'}

SPARK_TO_ONTOLOGY = {
    'string': 'String',
    'boolean': 'Boolean',
    'date': 'DateTime',
    'timestamp': 'DateTime',
    'double': 'Double',
    'float': 'Double',
    'byte': 'BigInt',
    'short': 'BigInt',
    'integer': 'BigInt',
    'int': 'BigInt',
    'long': 'BigInt',
    'bigint': 'BigInt',
    'smallint': 'BigInt',
    'tinyint': 'BigInt',
}

SOURCE_KIND_LAKEHOUSE = 'lakehouse'
SOURCE_KIND_EVENTHOUSE = 'eventhouse'
KUSTO_CONNECTOR_FORMAT = 'com.microsoft.kusto.spark.synapse.datasource'


def resolve_schema_db(cfg):
    marker = cfg.get('schema_db')
    if marker == 'gold':
        return GOLD_DB
    if marker == 'bronze':
        return BRONZE_DB
    return SILVER_DB


def get_source_kind(cfg):
    return cfg.get('source', SOURCE_KIND_LAKEHOUSE)


def is_eventhouse_source(cfg):
    return get_source_kind(cfg) == SOURCE_KIND_EVENTHOUSE


def build_table_name(table_name, schema=None):
    return f'{LAKEHOUSE_NAME}.{schema or SILVER_DB}.{table_name}'


def build_source_label(table_name, cfg, schema=None):
    if is_eventhouse_source(cfg):
        return f'{KQL_DATABASE_NAME}.{table_name}'
    return build_table_name(table_name, schema)


def snake_case(name):
    value = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    value = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', value)
    value = re.sub(r'[^0-9a-zA-Z]+', '_', value).strip('_')
    value = re.sub(r'_+', '_', value)
    return value.lower()


def make_unique_name(base_name, used_names):
    candidate = base_name
    suffix = 2
    while candidate in used_names:
        candidate = f'{base_name}_{suffix}'
        suffix += 1
    used_names.add(candidate)
    return candidate


def resolve_schema_column(field_names, candidates, label):
    lookup = {field.lower(): field for field in field_names}
    for candidate in candidates:
        resolved = lookup.get(candidate.lower())
        if resolved:
            return resolved
    raise KeyError(
        f'{label}: none of {candidates} were found. Available columns: {field_names}'
    )


def map_spark_type(data_type):
    simple_type = data_type.simpleString()
    if simple_type.startswith('decimal'):
        return 'Double'
    if simple_type.startswith(('array', 'map', 'struct', 'binary')):
        return None
    return SPARK_TO_ONTOLOGY.get(simple_type)


_used_ids = set()
random.seed(42)


def generate_id():
    while True:
        id_val = random.randint(10**12, 10**15)
        if id_val not in _used_ids:
            _used_ids.add(id_val)
            return str(id_val)


def to_base64(document):
    if isinstance(document, (dict, list)):
        payload = json.dumps(document).encode('utf-8')
    else:
        payload = str(document).encode('utf-8')
    return base64.b64encode(payload).decode('utf-8')


In [ ]:
# =============================================================================
# FABRIC CONTEXT
# =============================================================================

client = fabric.FabricRestClient()
WORKSPACE_ID = fabric.get_notebook_workspace_id()

WORKSPACE_NAME = WORKSPACE_ID
EVENTHOUSE_ID = get_env('EVENTHOUSE_ID')
KQL_DATABASE_ID = get_env('KQL_DATABASE_ID')
_EVENTHOUSE_CONTEXT_RESOLVED = False


def _list_workspace_items(collection_name):
    response = client.get(f'v1/workspaces/{WORKSPACE_ID}/{collection_name}')
    response.raise_for_status()
    return response.json().get('value', [])


def _resolve_named_item(items, display_name, item_label):
    if not items:
        raise ValueError(f'No {item_label.lower()}s were found in workspace {WORKSPACE_NAME}')
    if display_name:
        matches = [
            item for item in items
            if item.get('displayName', '').lower() == display_name.lower()
        ]
        if matches:
            return matches[0]
        available = ', '.join(sorted(item.get('displayName', '<unnamed>') for item in items))
        raise ValueError(
            f'{item_label} {display_name!r} was not found in workspace {WORKSPACE_NAME}. '
            f'Available {item_label.lower()}s: {available}'
        )
    if len(items) == 1:
        return items[0]
    available = ', '.join(sorted(item.get('displayName', '<unnamed>') for item in items))
    raise ValueError(
        f'{item_label} name was not provided and multiple candidates were found. '
        f'Set the matching environment variable to one of: {available}'
    )


def _get_kql_database_by_id(kql_database_id):
    response = client.get(f'v1/workspaces/{WORKSPACE_ID}/kqlDatabases/{kql_database_id}')
    response.raise_for_status()
    return response.json()


def _resolve_eventhouse_context():
    global EVENTHOUSE_ID, EVENTHOUSE_NAME, KQL_DATABASE_ID, KQL_DATABASE_NAME, KUSTO_URI
    global _EVENTHOUSE_CONTEXT_RESOLVED

    if _EVENTHOUSE_CONTEXT_RESOLVED:
        return
    if not EVENTHOUSE_ID:
        eventhouse = _resolve_named_item(
            _list_workspace_items('eventhouses'),
            EVENTHOUSE_NAME,
            'Eventhouse',
        )
        EVENTHOUSE_ID = eventhouse['id']
        EVENTHOUSE_NAME = eventhouse.get('displayName', EVENTHOUSE_NAME)
    if KQL_DATABASE_ID:
        kql_database = _get_kql_database_by_id(KQL_DATABASE_ID)
    else:
        kql_database = _resolve_named_item(
            _list_workspace_items('kqlDatabases'),
            KQL_DATABASE_NAME,
            'KQL database',
        )
        KQL_DATABASE_ID = kql_database['id']
    KQL_DATABASE_NAME = kql_database.get('displayName', KQL_DATABASE_NAME)
    if not KUSTO_URI:
        KUSTO_URI = (kql_database.get('properties') or {}).get('queryServiceUri', '')
    if not KUSTO_URI:
        raise ValueError(
            f'KQL database {KQL_DATABASE_NAME!r} has no queryServiceUri. '
            'Set KUSTO_URI to the Eventhouse Query URI.'
        )
    _EVENTHOUSE_CONTEXT_RESOLVED = True
    print(f'Eventhouse: {EVENTHOUSE_NAME} ({EVENTHOUSE_ID})')
    print(f'KQL database: {KQL_DATABASE_NAME} ({KQL_DATABASE_ID})')
    print(f'Kusto URI: {KUSTO_URI}')


lakehouse = _resolve_named_item(
    _list_workspace_items('lakehouses'),
    LAKEHOUSE_NAME,
    'Lakehouse',
)

LAKEHOUSE_ID = lakehouse['id']
LAKEHOUSE_NAME = lakehouse['displayName']

print(f'Workspace: {WORKSPACE_NAME} ({WORKSPACE_ID})')
print(f'Lakehouse: {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})')


In [ ]:
# =============================================================================
# READ SCHEMAS AND BUILD ONTOLOGY DEFINITION
# =============================================================================

def read_source_schema_fields(table_name, cfg, schema_db):
    if is_eventhouse_source(cfg):
        _resolve_eventhouse_context()
        query = f"table('{table_name}') | take 0"
        return (
            spark.read
            .format(KUSTO_CONNECTOR_FORMAT)
            .option('kustoCluster', KUSTO_URI)
            .option('kustoDatabase', KQL_DATABASE_NAME)
            .option('kustoQuery', query)
            .load()
            .schema
            .fields
        )
    return spark.table(build_table_name(table_name, schema_db)).schema.fields


def make_source_table_properties(table_name, source_kind, schema_db):
    if source_kind == SOURCE_KIND_EVENTHOUSE:
        _resolve_eventhouse_context()
        return {
            'sourceType': 'KustoTable',
            'workspaceId': WORKSPACE_ID,
            'itemId': EVENTHOUSE_ID,
            'clusterUri': KUSTO_URI,
            'databaseName': KQL_DATABASE_NAME,
            'sourceTableName': table_name,
        }
    return {
        'sourceType': 'LakehouseTable',
        'workspaceId': WORKSPACE_ID,
        'itemId': LAKEHOUSE_ID,
        'sourceTableName': table_name,
        'sourceSchema': schema_db,
    }


def compatible_ontology_columns(schema_fields):
    used_property_names = set()
    compatible_columns = []
    skipped_columns = []
    for field in schema_fields:
        if field.name in EXCLUDED_SOURCE_COLUMNS:
            skipped_columns.append(field.name)
            continue
        value_type = map_spark_type(field.dataType)
        if value_type is None:
            skipped_columns.append(field.name)
            continue
        compatible_columns.append({
            'source_name': field.name,
            'name': make_unique_name(snake_case(field.name), used_property_names),
            'valueType': value_type,
        })
    return compatible_columns, skipped_columns


def prefixed_timeseries_columns(table_name, schema_fields, excluded_source_names):
    used_property_names = set()
    columns = []
    skipped_columns = []
    for field in schema_fields:
        if field.name in excluded_source_names or field.name in EXCLUDED_SOURCE_COLUMNS:
            skipped_columns.append(field.name)
            continue
        value_type = map_spark_type(field.dataType)
        if value_type is None:
            skipped_columns.append(field.name)
            continue
        property_key = f"{table_name}.{field.name}"
        base_name = f"{snake_case(table_name)}_{snake_case(field.name)}"
        columns.append({
            'source_name': field.name,
            'property_key': property_key,
            'name': make_unique_name(base_name, used_property_names),
            'valueType': value_type,
        })
    return columns, skipped_columns


def resolve_table_field_names(table_name, cfg, schema_db):
    full_table_name = build_source_label(table_name, cfg, schema_db)
    if full_table_name not in table_field_names:
        schema_fields = read_source_schema_fields(table_name, cfg, schema_db)
        table_field_names[full_table_name] = [field.name for field in schema_fields]
        table_schema_fields[full_table_name] = schema_fields
    return full_table_name, table_field_names[full_table_name], table_schema_fields[full_table_name]


table_field_names = {}
table_schema_fields = {}
entity_type_ids = {}
property_ids = {}
key_property_ids = {}
entity_metadata = {}
relationship_metadata = []
parts = [
    {
        'path': '.platform',
        'payload': to_base64({
            'metadata': {'type': 'Ontology', 'displayName': ONTOLOGY_NAME}
        }),
        'payloadType': 'InlineBase64',
    },
    {'path': 'definition.json', 'payload': to_base64({}), 'payloadType': 'InlineBase64'},
]

print('Resolving business entity schemas...')
for table_name, cfg in ENTITY_CONFIG.items():
    schema_db = resolve_schema_db(cfg)
    source_kind = get_source_kind(cfg)
    full_table_name = build_source_label(table_name, cfg, schema_db)
    try:
        schema_fields = read_source_schema_fields(table_name, cfg, schema_db)
    except Exception as exc:  # noqa: BLE001 - optional tables may not exist yet
        if cfg.get('optional'):
            print(f"  - skipping optional entity {cfg['entity_name']} ({exc})")
            continue
        raise
    field_names = [field.name for field in schema_fields]
    table_field_names[full_table_name] = field_names
    table_schema_fields[full_table_name] = schema_fields

    key_column = resolve_schema_column(
        field_names,
        cfg['key_candidates'],
        f'{full_table_name} key column',
    )
    try:
        display_column = resolve_schema_column(
            field_names,
            cfg['display_candidates'],
            f'{full_table_name} display column',
        )
    except KeyError:
        display_column = key_column

    compatible_columns, skipped_columns = compatible_ontology_columns(schema_fields)
    compatible_source_names = {column['source_name'] for column in compatible_columns}
    if key_column not in compatible_source_names:
        raise ValueError(f'{full_table_name} key column {key_column} has an unsupported ontology type')
    if display_column not in compatible_source_names:
        display_column = key_column

    entity_metadata[cfg['entity_name']] = {
        'table_name': table_name,
        'schema_db': schema_db,
        'source_kind': source_kind,
        'full_table_name': full_table_name,
        'key_column': key_column,
        'display_column': display_column,
        'columns': compatible_columns,
        'timeseries_columns': [],
        'eventhouse_bindings': [],
    }
    print(
        f"  - {cfg['entity_name']}: table={full_table_name}, properties={len(compatible_columns)}, "
        f"key={key_column}, display={display_column}, skipped={len(skipped_columns)}"
    )

print()
print('Resolving Eventhouse TimeSeries bindings on business entities...')
for binding in EVENTHOUSE_ENTITY_BINDINGS:
    entity_name = binding['entity_name']
    if entity_name not in entity_metadata:
        print(f"  - skipping Eventhouse binding for {entity_name} (entity not resolved)")
        continue
    schema_db = resolve_schema_db(binding)
    try:
        full_table_name, field_names, schema_fields = resolve_table_field_names(
            binding['source_table'],
            binding,
            schema_db,
        )
    except Exception as exc:  # noqa: BLE001 - optional Eventhouse tables may not exist yet
        if binding.get('optional'):
            print(f"  - skipping optional Eventhouse binding {entity_name} <- {binding['source_table']} ({exc})")
            continue
        raise
    source_key_column = resolve_schema_column(
        field_names,
        binding['source_column_candidates'],
        f"{full_table_name} key column for {entity_name}",
    )
    timestamp_column = resolve_schema_column(
        field_names,
        binding.get('timestamp_candidates', ['ingest_timestamp']),
        f"{full_table_name} timestamp column for {entity_name}",
    )
    timeseries_columns, skipped_columns = prefixed_timeseries_columns(
        binding['source_table'],
        schema_fields,
        excluded_source_names={source_key_column},
    )
    entity_metadata[entity_name]['timeseries_columns'].extend(timeseries_columns)
    entity_metadata[entity_name]['eventhouse_bindings'].append({
        'table_name': binding['source_table'],
        'schema_db': schema_db,
        'source_kind': get_source_kind(binding),
        'full_table_name': full_table_name,
        'source_key_column': source_key_column,
        'timestamp_column': timestamp_column,
        'timeseries_columns': timeseries_columns,
    })
    print(
        f"  - {entity_name}: Eventhouse table={full_table_name}, key={source_key_column}, "
        f"timeseries={len(timeseries_columns)}, skipped={len(skipped_columns)}"
    )

print()
print('Resolving relationship contextualizations...')
for rel in list(RELATIONSHIPS) + list(EVENTHOUSE_RELATIONSHIP_CONTEXTS):
    if rel['source_entity'] not in entity_metadata or rel['target_entity'] not in entity_metadata:
        print(f"  - skipping relationship {rel['name']} (entity not resolved)")
        continue
    rel_schema_db = resolve_schema_db(rel)
    try:
        rel_full_table_name, source_field_names, _ = resolve_table_field_names(
            rel['source_table'],
            rel,
            rel_schema_db,
        )
    except Exception as exc:  # noqa: BLE001 - optional tables may not exist yet
        if rel.get('optional'):
            print(f"  - skipping optional relationship {rel['name']} ({exc})")
            continue
        raise

    resolved_source_column = resolve_schema_column(
        source_field_names,
        rel['source_column_candidates'],
        f"{rel_full_table_name} source column for {rel['name']}",
    )
    resolved_target_column = resolve_schema_column(
        source_field_names,
        rel['target_column_candidates'],
        f"{rel_full_table_name} target column for {rel['name']}",
    )
    relationship_metadata.append({
        'name': rel['name'],
        'source_table': rel['source_table'],
        'schema_db': rel_schema_db,
        'source_kind': get_source_kind(rel),
        'full_table_name': rel_full_table_name,
        'source_entity': rel['source_entity'],
        'target_entity': rel['target_entity'],
        'source_column': resolved_source_column,
        'target_column': resolved_target_column,
    })
    print(
        f"  - {rel['name']}: {rel['source_entity']}->{rel['target_entity']} "
        f"via {rel_full_table_name}({resolved_source_column}, {resolved_target_column})"
    )

print()
print('Building ontology definition parts...')
for entity_name, metadata in entity_metadata.items():
    entity_type_id = generate_id()
    entity_type_ids[entity_name] = entity_type_id

    properties = []
    for column in metadata['columns']:
        prop_id = generate_id()
        property_ids[(entity_name, 'properties', column['source_name'])] = prop_id
        properties.append({
            'id': prop_id,
            'name': column['name'],
            'redefines': None,
            'baseTypeNamespaceType': None,
            'valueType': column['valueType'],
        })

    timeseries_properties = []
    for column in metadata['timeseries_columns']:
        prop_id = generate_id()
        prop_key = column.get('property_key', column['source_name'])
        property_ids[(entity_name, 'timeseries', prop_key)] = prop_id
        timeseries_properties.append({
            'id': prop_id,
            'name': column['name'],
            'redefines': None,
            'baseTypeNamespaceType': None,
            'valueType': column['valueType'],
        })

    key_prop_id = property_ids[(entity_name, 'properties', metadata['key_column'])]
    key_property_ids[entity_name] = key_prop_id
    display_prop_id = property_ids[(entity_name, 'properties', metadata['display_column'])]
    entity_def = {
        'id': entity_type_id,
        'namespace': 'usertypes',
        'baseEntityTypeId': None,
        'name': entity_name,
        'entityIdParts': [key_prop_id],
        'displayNamePropertyId': display_prop_id,
        'namespaceType': 'Custom',
        'visibility': 'Visible',
        'properties': properties,
        'timeseriesProperties': timeseries_properties,
    }
    parts.append({
        'path': f'EntityTypes/{entity_type_id}/definition.json',
        'payload': to_base64(entity_def),
        'payloadType': 'InlineBase64',
    })

    base_property_bindings = [
        {
            'sourceColumnName': column['source_name'],
            'targetPropertyId': property_ids[(entity_name, 'properties', column['source_name'])],
        }
        for column in metadata['columns']
    ]
    base_data_binding = {
        'id': str(uuid.uuid4()),
        'dataBindingConfiguration': {
            'dataBindingType': 'NonTimeSeries',
            'propertyBindings': base_property_bindings,
            'sourceTableProperties': make_source_table_properties(
                metadata['table_name'],
                metadata['source_kind'],
                metadata['schema_db'],
            ),
        },
    }
    parts.append({
        'path': f"EntityTypes/{entity_type_id}/DataBindings/{base_data_binding['id']}.json",
        'payload': to_base64(base_data_binding),
        'payloadType': 'InlineBase64',
    })

    for event_binding in metadata['eventhouse_bindings']:
        property_bindings = [
            {
                'sourceColumnName': event_binding['source_key_column'],
                'targetPropertyId': key_prop_id,
            }
        ]
        property_bindings.extend(
            {
                'sourceColumnName': column['source_name'],
                'targetPropertyId': property_ids[(
                    entity_name,
                    'timeseries',
                    column.get('property_key', column['source_name']),
                )],
            }
            for column in event_binding['timeseries_columns']
        )
        data_binding = {
            'id': str(uuid.uuid4()),
            'dataBindingConfiguration': {
                'dataBindingType': 'TimeSeries',
                'timestampColumnName': event_binding['timestamp_column'],
                'propertyBindings': property_bindings,
                'sourceTableProperties': make_source_table_properties(
                    event_binding['table_name'],
                    event_binding['source_kind'],
                    event_binding['schema_db'],
                ),
            },
        }
        parts.append({
            'path': f"EntityTypes/{entity_type_id}/DataBindings/{data_binding['id']}.json",
            'payload': to_base64(data_binding),
            'payloadType': 'InlineBase64',
        })

    print(
        f"  - Entity {entity_name}: {len(properties)} properties, "
        f"{len(timeseries_properties)} timeseries properties, "
        f"{1 + len(metadata['eventhouse_bindings'])} data bindings"
    )

relationship_groups = {}
for rel in relationship_metadata:
    key = (rel['name'], rel['source_entity'], rel['target_entity'])
    relationship_groups.setdefault(key, []).append(rel)

for (rel_name, source_entity, target_entity), contexts in relationship_groups.items():
    rel_id = generate_id()
    rel_def = {
        'namespace': 'usertypes',
        'id': rel_id,
        'name': rel_name,
        'namespaceType': 'Custom',
        'source': {'entityTypeId': entity_type_ids[source_entity]},
        'target': {'entityTypeId': entity_type_ids[target_entity]},
    }
    parts.append({
        'path': f'RelationshipTypes/{rel_id}/definition.json',
        'payload': to_base64(rel_def),
        'payloadType': 'InlineBase64',
    })

    for rel in contexts:
        ctx_id = str(uuid.uuid4())
        contextualization = {
            'id': ctx_id,
            'dataBindingTable': make_source_table_properties(
                rel['source_table'],
                rel['source_kind'],
                rel['schema_db'],
            ),
            'sourceKeyRefBindings': [
                {
                    'sourceColumnName': rel['source_column'],
                    'targetPropertyId': key_property_ids[source_entity],
                }
            ],
            'targetKeyRefBindings': [
                {
                    'sourceColumnName': rel['target_column'],
                    'targetPropertyId': key_property_ids[target_entity],
                }
            ],
        }
        parts.append({
            'path': f'RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json',
            'payload': to_base64(contextualization),
            'payloadType': 'InlineBase64',
        })
    print(
        f"  - Relationship {rel_name}: {source_entity}->{target_entity}, "
        f"contextualizations={len(contexts)}"
    )

print()
print(
    f'Prepared {len(entity_metadata)} entity types, {len(relationship_groups)} relationships, '
    f'{len(parts)} total parts.'
)


In [ ]:
# =============================================================================
# CREATE OR UPDATE THE ONTOLOGY ITEM
# =============================================================================
#
# Prefer UPDATING an existing ontology in place over delete+recreate. Fabric
# keeps a deleted item's display name reserved for a short cooldown, so an
# immediate recreate with the same name fails on the first attempt and only
# succeeds on a later retry -- the "ontology ran twice (errored, then
# succeeded)" symptom. updateDefinition replaces the definition without
# deleting, so there is no name cooldown and no first-attempt failure.
# Delete+create stays as a fallback in case in-place update is ever rejected, so
# this is never worse than a plain recreate.

definition = {'parts': parts}


def _find_ontology_id():
    """Return the id of the ontology named ONTOLOGY_NAME, or None if absent."""
    listing = client.get(f'v1/workspaces/{WORKSPACE_ID}/ontologies')
    listing.raise_for_status()
    return next(
        (o['id'] for o in listing.json().get('value', [])
         if o.get('displayName') == ONTOLOGY_NAME),
        None,
    )


def _poll_operation(operation_id, retry_after):
    """Poll a Fabric long-running operation until it terminates; return status."""
    while True:
        time.sleep(retry_after)
        poll_response = client.get(f'v1/operations/{operation_id}')
        poll_response.raise_for_status()
        status = poll_response.json().get('status')
        print(f'  Status: {status}')
        if status in ('Succeeded', 'Failed', 'Cancelled'):
            if status != 'Succeeded':
                print(f'  Operation detail: {poll_response.json()}')
            return status


def _update_in_place(ontology_id):
    """Replace the existing ontology's definition without deleting it.

    Returns True on success, False if the API rejects in-place update (so the
    caller can fall back to delete + recreate).
    """
    print(f"Updating existing ontology '{ONTOLOGY_NAME}' (id={ontology_id}) in place...")
    # `parts` includes a .platform metadata part, so updateMetadata=true is
    # required (omitting it with a .platform part returns HTTP 400).
    response = client.post(
        f'v1/workspaces/{WORKSPACE_ID}/ontologies/{ontology_id}/updateDefinition?updateMetadata=true',
        json={'definition': definition},
    )
    if response.status_code in (200, 201):
        print('Ontology updated.')
        return True
    if response.status_code == 202:
        operation_id = response.headers.get('x-ms-operation-id')
        retry_after = int(response.headers.get('Retry-After', '30'))
        print(f'Operation {operation_id} accepted. Polling every {retry_after} seconds...')
        if _poll_operation(operation_id, retry_after) == 'Succeeded':
            print('Ontology updated.')
            return True
        return False
    print(f'  In-place update not accepted (HTTP {response.status_code}); will recreate.')
    return False


def _create_new():
    """Create the ontology. Returns True on success, False on a retryable conflict."""
    create_response = client.post(
        f'v1/workspaces/{WORKSPACE_ID}/ontologies',
        json={
            'displayName': ONTOLOGY_NAME,
            'description': ONTOLOGY_DESCRIPTION,
            'definition': definition,
        },
    )
    if create_response.status_code == 201:
        print(f"Ontology created. ID: {create_response.json().get('id', 'N/A')}")
        return True
    if create_response.status_code == 202:
        operation_id = create_response.headers.get('x-ms-operation-id')
        retry_after = int(create_response.headers.get('Retry-After', '30'))
        print(f'Operation {operation_id} accepted. Polling every {retry_after} seconds...')
        if _poll_operation(operation_id, retry_after) == 'Succeeded':
            print('Ontology created.')
            return True
        return False
    if create_response.status_code == 409:
        # Name still reserved by the just-deleted ontology; retry after a wait.
        print('  Ontology name still in use after delete; will retry.')
        return False
    print(f'HTTP {create_response.status_code}: {create_response.text}')
    create_response.raise_for_status()
    return False


def _create_with_retries():
    """Create the ontology, retrying to ride out the display-name cooldown."""
    for attempt in range(1, 6):
        if _create_new():
            return
        if attempt == 5:
            raise RuntimeError('Ontology creation did not succeed after 5 attempts.')
        print(f'  Create attempt {attempt} did not succeed; retrying in {DELETE_WAIT_SECONDS}s...')
        time.sleep(DELETE_WAIT_SECONDS)


def _delete_then_create():
    """Fallback: delete the existing ontology, wait for it to disappear, recreate."""
    existing = _find_ontology_id()
    if existing:
        print(f"Deleting existing ontology '{ONTOLOGY_NAME}' (id={existing})...")
        client.delete(f'v1/workspaces/{WORKSPACE_ID}/ontologies/{existing}').raise_for_status()
        deadline = time.time() + max(DELETE_WAIT_SECONDS, 30) * 10
        while _find_ontology_id() is not None:
            if time.time() > deadline:
                raise RuntimeError(
                    f"Existing ontology '{ONTOLOGY_NAME}' was not deleted within the timeout."
                )
            print(f'Waiting {DELETE_WAIT_SECONDS}s for Fabric to finish deleting the existing ontology...')
            time.sleep(DELETE_WAIT_SECONDS)
    _create_with_retries()


existing_id = _find_ontology_id()

if existing_id and not DELETE_EXISTING:
    raise ValueError(
        f"Ontology '{ONTOLOGY_NAME}' already exists. Set DELETE_EXISTING=true to replace it."
    )

if existing_id:
    if not _update_in_place(existing_id):
        print('In-place update failed; falling back to delete + recreate.')
        _delete_then_create()
else:
    print(f"Creating ontology '{ONTOLOGY_NAME}'...")
    _create_with_retries()